# LLM Fine-Tuning Training Loop From Scratch + Ollama Evaluation

This notebook shows a practical fine-tuning workflow for a causal language model using a **manual PyTorch training loop**. It avoids high-level trainer classes so the core mechanics are visible: tokenization, batching, forward pass, loss, backpropagation, optimizer steps, validation, generation, checkpointing, and evaluation.

The final section uses **Ollama** as a local evaluator. The fine-tuned model generates answers, then an Ollama-hosted model scores those answers using a rubric.

> Note: This is a training-loop-from-scratch notebook, not a transformer-architecture-from-scratch notebook. We start from a small pretrained causal LM so the notebook can run on a normal laptop or Colab session.

## 1. Install Dependencies

Run this cell if the packages are not already installed. Restart the kernel after installation if your notebook environment asks you to.

In [1]:
# Uncomment when needed.
# %pip install torch transformers datasets pandas tqdm requests

## 2. Imports and Configuration

We keep the model small by default. `distilgpt2` is not a strong instruction model, but it is lightweight enough for demonstrating the full loop. For better results, replace it with a stronger causal LM that fits your hardware.

In [2]:
import json
import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path

import requests
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "distilgpt2"
OUTPUT_DIR = Path("fine_tuned_distilgpt2_instruction_demo")
MAX_LENGTH = 192
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
EPOCHS = 3
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
MAX_NEW_TOKENS = 80

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## 3. Build a Small Instruction Dataset

Fine-tuning needs examples in a consistent format. Each example below has an instruction, optional input, and expected response. In real projects, replace this list with a larger JSONL/CSV dataset.

The formatting matters because the model learns the pattern. We use a simple prompt template:

```text
### Instruction:
...

### Input:
...

### Response:
...
```

In [3]:
examples = [
    {
        "instruction": "Explain gradient descent in simple terms.",
        "input": "",
        "response": "Gradient descent is a way to improve a model by making small changes in the direction that reduces error. It measures how wrong the model is, calculates which parameters caused the error, and nudges those parameters to make the next prediction better."
    },
    {
        "instruction": "Convert the sentence into a professional tone.",
        "input": "send me the report today or we will have problems",
        "response": "Please send me the report today so we can avoid delays and keep the project on track."
    },
    {
        "instruction": "Classify the sentiment as positive, neutral, or negative.",
        "input": "The delivery was late, but the support team fixed the issue quickly.",
        "response": "neutral"
    },
    {
        "instruction": "Summarize the paragraph in one sentence.",
        "input": "Large language models are trained on huge text datasets. They learn statistical patterns in language and can generate fluent responses, but they can also make mistakes or invent facts.",
        "response": "Large language models generate fluent text from learned patterns, but their outputs still need verification."
    },
    {
        "instruction": "Write a Python function that returns the square of a number.",
        "input": "",
        "response": "def square(x):\n    return x * x"
    },
    {
        "instruction": "Explain overfitting.",
        "input": "",
        "response": "Overfitting happens when a model memorizes training examples too closely and performs poorly on new data. It usually means the model learned noise or accidental details instead of general patterns."
    },
    {
        "instruction": "Extract the action item.",
        "input": "During the meeting, Priya said she would prepare the launch checklist before Friday.",
        "response": "Priya should prepare the launch checklist before Friday."
    },
    {
        "instruction": "Give two ways to reduce hallucinations in LLM outputs.",
        "input": "",
        "response": "Use retrieval from trusted sources and require the model to cite or quote evidence. Also add evaluation checks that compare answers against known facts."
    },
]

def format_example(example, include_response=True):
    text = f"### Instruction:\n{example['instruction'].strip()}\n\n"
    if example.get("input", "").strip():
        text += f"### Input:\n{example['input'].strip()}\n\n"
    text += "### Response:\n"
    if include_response:
        text += example["response"].strip()
    return text

formatted_examples = [format_example(ex) for ex in examples]
print(formatted_examples[0])

### Instruction:
Explain gradient descent in simple terms.

### Response:
Gradient descent is a way to improve a model by making small changes in the direction that reduces error. It measures how wrong the model is, calculates which parameters caused the error, and nudges those parameters to make the next prediction better.


## 4. Train/Validation Split

The dataset is tiny for demonstration, so validation numbers are only a sanity check. With real fine-tuning, use a larger validation set that represents the actual task distribution.

In [4]:
random.shuffle(formatted_examples)
split_idx = max(1, int(0.75 * len(formatted_examples)))
train_texts = formatted_examples[:split_idx]
val_texts = formatted_examples[split_idx:]

len(train_texts), len(val_texts)

(6, 2)

## 5. Tokenizer and Model

Causal language models predict the next token. During fine-tuning, the labels are usually the same as the input IDs, shifted internally by the model loss function.

`distilgpt2` does not define a padding token, so we reuse the end-of-text token for padding.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Model parameters: 81.9M


## 6. Custom Dataset and Collate Function

The dataset returns tokenized tensors. The collate function pads each batch dynamically. Labels use `-100` on padding positions so the loss ignores padded tokens.

In [6]:
class InstructionDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            add_special_tokens=True,
        )
        return {"input_ids": encoded["input_ids"], "attention_mask": encoded["attention_mask"]}

def collate_batch(batch):
    max_len = max(len(item["input_ids"]) for item in batch)
    input_ids, attention_masks, labels = [], [], []

    for item in batch:
        pad_len = max_len - len(item["input_ids"])
        ids = item["input_ids"] + [tokenizer.pad_token_id] * pad_len
        mask = item["attention_mask"] + [0] * pad_len
        label = ids.copy()
        label = [tok if m == 1 else -100 for tok, m in zip(label, mask)]

        input_ids.append(ids)
        attention_masks.append(mask)
        labels.append(label)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

train_loader = DataLoader(
    InstructionDataset(train_texts, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    InstructionDataset(val_texts, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch.items()}

{'input_ids': torch.Size([2, 49]),
 'attention_mask': torch.Size([2, 49]),
 'labels': torch.Size([2, 49])}

## 7. Evaluation Function

Perplexity is the exponent of average cross-entropy loss. Lower perplexity generally means the model predicts validation text better, but it does not guarantee better instruction-following quality.

In [7]:
@torch.no_grad()
def evaluate_loss(model, dataloader):
    model.eval()
    total_loss = 0.0
    total_batches = 0

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item()
        total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    perplexity = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    return avg_loss, perplexity

initial_val_loss, initial_val_ppl = evaluate_loss(model, val_loader)
print(f"Initial validation loss: {initial_val_loss:.4f}")
print(f"Initial validation perplexity: {initial_val_ppl:.2f}")

Initial validation loss: 4.0238
Initial validation perplexity: 55.92


## 8. Manual Fine-Tuning Loop

This is the core loop:

1. Put the model in training mode.
2. Move a batch to the active device.
3. Run the forward pass and compute loss.
4. Backpropagate with `loss.backward()`.
5. Step the optimizer after optional gradient accumulation.
6. Validate at the end of each epoch.

Gradient accumulation lets us simulate a larger batch size when memory is limited.

In [8]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = []
global_step = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for step, batch in enumerate(progress, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()

        running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS

        should_step = step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader)
        if should_step:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        progress.set_postfix({"loss": f"{running_loss / step:.4f}"})

    train_loss = running_loss / max(len(train_loader), 1)
    val_loss, val_ppl = evaluate_loss(model, val_loader)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_ppl": val_ppl})
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_ppl={val_ppl:.2f}")

history

Epoch 1/3:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 1: train_loss=4.7877, val_loss=3.8142, val_ppl=45.34


Epoch 2/3:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 2: train_loss=4.4029, val_loss=3.6541, val_ppl=38.63


Epoch 3/3:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 3: train_loss=4.0682, val_loss=3.5354, val_ppl=34.31


[{'epoch': 1,
  'train_loss': 4.787699540456136,
  'val_loss': 3.814178466796875,
  'val_ppl': 45.33949317950378},
 {'epoch': 2,
  'train_loss': 4.4028990268707275,
  'val_loss': 3.654069423675537,
  'val_ppl': 38.63155477279478},
 {'epoch': 3,
  'train_loss': 4.06817642847697,
  'val_loss': 3.535374641418457,
  'val_ppl': 34.307865454246816}]

## 9. Generate With the Fine-Tuned Model

For instruction-style generation, pass the prompt without the response. The model should continue after `### Response:`.

In [9]:
@torch.no_grad()
def generate_response(model, tokenizer, instruction, input_text="", max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    prompt = format_example({"instruction": instruction, "input": input_text, "response": ""}, include_response=False)
    encoded = tokenizer(prompt, return_tensors="pt").to(device)

    output_ids = model.generate(
        **encoded,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return decoded[len(prompt):].strip()

test_instruction = "Explain why validation data is important in machine learning."
print(generate_response(model, tokenizer, test_instruction))

The answer is:
We need to implement validation data to make machine learning more accurate.


### Response:
If we need validation data to make machine learning more accurate, we can use validation data as a validation tool.

### Response:
For example, if we use validation data to make machine learning more accurate, we can use validation data as a validation tool.
###


## 10. Save and Reload the Fine-Tuned Model

This saves model weights and tokenizer files in Hugging Face format. You can reload them later with `from_pretrained`.

In [10]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(OUTPUT_DIR / "training_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

print(f"Saved fine-tuned model to: {OUTPUT_DIR.resolve()}")

Saved fine-tuned model to: D:\Data Science\LLMs\fine_tuned_distilgpt2_instruction_demo


## 11. Evaluate the Fine-Tuned Model With Ollama

Ollama can run a local evaluator model through an HTTP API. This notebook uses Ollama as an **LLM judge**. The flow is:

1. Generate an answer using the fine-tuned model.
2. Send the instruction, optional input, reference answer, and model answer to Ollama.
3. Ask Ollama to return structured JSON scores.

Before running the cells below, install Ollama and pull a judge model in a terminal:

```bash
ollama pull llama3.1:8b
ollama serve
```

If you already have another local model, change `OLLAMA_JUDGE_MODEL`.

In [11]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_JUDGE_MODEL = "gemma3:latest"

eval_examples = [
    {
        "instruction": "Explain early stopping in machine learning.",
        "input": "",
        "reference": "Early stopping stops training when validation performance stops improving, which helps prevent overfitting."
    },
    {
        "instruction": "Rewrite this in a polite tone.",
        "input": "you forgot to send the dataset again",
        "reference": "Could you please send the dataset again when you have a chance?"
    },
    {
        "instruction": "Classify the sentiment as positive, neutral, or negative.",
        "input": "The model is fast, but the results are unreliable.",
        "reference": "neutral"
    },
]

### Ollama Health Check

This checks whether Ollama is reachable. If it fails, start Ollama locally and make sure the model name is available.

In [12]:
def ollama_generate(prompt, model=OLLAMA_JUDGE_MODEL, temperature=0.0):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature},
    }
    response = requests.post(OLLAMA_URL, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["response"]

try:
    print(ollama_generate("Reply with exactly: Ollama is ready."))
except Exception as exc:
    print("Ollama check failed:", exc)

Ollama is ready.


### Judge Prompt

The judge scores each answer from 1 to 5 on correctness, helpfulness, and style. JSON output makes the result easier to aggregate.

In [13]:
def build_judge_prompt(item, candidate_answer):
    return f"""
You are evaluating the answer from a fine-tuned language model.

Score the candidate answer from 1 to 5 for each criterion:
- correctness: factual accuracy and alignment with the reference
- helpfulness: usefulness and completeness for the user
- style: clarity, tone, and directness

Return only valid JSON with this schema:
{{
  "correctness": number,
  "helpfulness": number,
  "style": number,
  "overall": number,
  "reason": "short explanation"
}}

Instruction: {item['instruction']}
Input: {item.get('input', '')}
Reference answer: {item['reference']}
Candidate answer: {candidate_answer}
""".strip()

def parse_json_safely(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}") + 1
        if start >= 0 and end > start:
            return json.loads(text[start:end])
        raise

def evaluate_with_ollama(eval_items):
    results = []
    for item in tqdm(eval_items, desc="Ollama evaluation"):
        candidate = generate_response(model, tokenizer, item["instruction"], item.get("input", ""))
        judge_prompt = build_judge_prompt(item, candidate)
        raw_judgment = ollama_generate(judge_prompt)

        try:
            judgment = parse_json_safely(raw_judgment)
        except Exception:
            judgment = {"raw_judgment": raw_judgment}

        results.append({
            "instruction": item["instruction"],
            "input": item.get("input", ""),
            "reference": item["reference"],
            "candidate": candidate,
            "judgment": judgment,
        })
    return results

In [14]:
# Run this after the Ollama health check succeeds.
# ollama_results = evaluate_with_ollama(eval_examples)
# ollama_results

## 12. Aggregate Ollama Scores

When the judge returns valid JSON, this cell averages the scores. Treat these numbers as a helpful signal, not absolute truth. For serious evaluation, combine LLM-as-judge scoring with human review and task-specific metrics.

In [15]:
def summarize_ollama_results(results):
    numeric_keys = ["correctness", "helpfulness", "style", "overall"]
    summary = {}

    for key in numeric_keys:
        values = []
        for row in results:
            value = row.get("judgment", {}).get(key)
            if isinstance(value, (int, float)):
                values.append(float(value))
        summary[key] = sum(values) / len(values) if values else None

    return summary

# summarize_ollama_results(ollama_results)

## 13. Practical Improvements

- Use a larger and cleaner instruction dataset.
- Mask prompt tokens so the loss focuses only on the response.
- Use LoRA/QLoRA for larger models when GPU memory is limited.
- Track experiments with validation loss, held-out prompts, and human inspection.
- Use task-specific metrics where possible, such as accuracy for classification or exact match for structured extraction.
- Keep an untouched test set for final evaluation.

This notebook is intentionally small so every moving part remains visible. The same training-loop structure scales to larger datasets and stronger base models.

## 14. What Is the Generated Model Folder?

After running the save cell, the notebook creates this folder:

```text
fine_tuned_distilgpt2_instruction_demo/
```

This folder contains the saved version of the fine-tuned model. It is created by these lines:

```python
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
```

Common files inside the folder include:

- `model.safetensors`: the fine-tuned model weights.
- `config.json`: the model architecture and configuration.
- `generation_config.json`: default text-generation settings.
- `tokenizer.json`: the tokenizer saved in a compact format.
- `tokenizer_config.json`: tokenizer settings used during loading.
- `vocab.json`: the tokenizer vocabulary.
- `merges.txt`: byte-pair encoding merge rules used by GPT-style tokenizers.
- `special_tokens_map.json`: special token definitions such as padding or end-of-text tokens.
- `training_history.json`: the training and validation loss history saved by this notebook.

You can reload the fine-tuned model later with:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("fine_tuned_distilgpt2_instruction_demo")
tokenizer = AutoTokenizer.from_pretrained("fine_tuned_distilgpt2_instruction_demo")
```

Keep this folder if you want to reuse the fine-tuned model without training again. Delete it only if you no longer need the saved checkpoint.